# Experiment 14 — Revision analyses

Analyses requested by the stress test of the JOP manuscript (18 Oct 2026). Six modules, each exporting its own folder:

| Module | Purpose | Stress-test items |
|---|---|---|
| 14A | Simulated 21-day experimental-gingivitis rechallenge at graded times after therapy (reference and single-state sets); persistence and reversibility | 1c, 6 |
| 14B | Figure 3 rebuilt from a model-defined therapy (debridement + plaque control); durations reported; r = 0.03 check; relapse ceiling | 7, 10, 11, 14 |
| 14C | Ratchet with first inflamed recall and first damaging recall; latency vs passage time with N(1 − e^−εT); r = 0.03 check | 4, 7, 12, 13 |
| 14D | Episodic activity with per-site sequences, activity-independent clustering measures and activity-matched comparisons | 5 |
| 14E | Robustness sample in the bistable regime: stage boundaries, window ratio, ratchet margin over the instant host | 15 |
| 14F | Grid check of conversion monotonicity for the dose-dependent host | 16 |

Conventions follow Experiments 07–13: reference set n = q = 4, K = 0.9, θ0 = 0.8, r = 1, α = 1; fixed-step RK4, dt = 0.1;
conversion = H > 0.5 after settling; one time unit = 100/60 days (T = 60 ≈ 100 days).

**Before running:** optionally upload `Experiment_12_Episodic_Activity_EXPORT.zip` (or its `peak_sequences.npz`) so 14D reproduces
Experiment 12 exactly. Set `FAST = True` for a quick test run (minutes); `FAST = False` for the full run (roughly 1–2 h on Colab CPU).


In [ ]:
# ---------------------------------------------------------------- setup
import os, json, time, zipfile, platform, glob
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import brentq
from scipy.stats import qmc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FAST = False          # True: tiny grids for a smoke test
FAST = FAST or os.environ.get("EXP14_FAST") == "1"
RUN = dict(A=True, B=True, C=True, D=True, E=True, F=True)

OUT_ROOT = "Experiment_14_Revision_EXPORT"
os.makedirs(OUT_ROOT, exist_ok=True)

DT = 0.1
UNIT_DAYS = 100.0 / 60.0          # T = 60 units ~ 100 days
def days(d): return d / UNIT_DAYS
K_REGROW = 0.05
K_ACCESS = 0.5                     # fast (accessible) plaque regrowth, as in Exp 10
GAMMA, H_D = 0.8, 0.8

REF    = dict(n=4.0, q=4.0, K=0.9, th0=0.8, r=1.0, a=1.0)
SECOND = dict(n=4.0, q=4.0, K=0.5, th0=0.6, r=1.0, a=1.0)
SINGLE = dict(n=2.0, q=1.0, K=0.9, th0=1.0, r=1.0, a=1.0)

def outdir(name):
    d = os.path.join(OUT_ROOT, name); os.makedirs(d, exist_ok=True); return d

def save_json(d, name, obj):
    with open(os.path.join(d, name), "w") as f: json.dump(obj, f, indent=2, default=float)

def save_versions(d):
    with open(os.path.join(d, "software_versions.txt"), "w") as f:
        f.write(f"python {platform.python_version()}\nnumpy {np.__version__}\nscipy {scipy.__version__}\n"
                f"pandas {pd.__version__}\nmatplotlib {matplotlib.__version__}\nFAST {FAST}\n")

def save_checks(d, checks):
    pd.DataFrame(checks, columns=["check", "value", "passed"]).to_csv(os.path.join(d, "checks.csv"), index=False)

T0 = time.time()
def stamp(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)


In [ ]:
# ---------------------------------------------------------------- vectorized model
def hill(x, K, p):
    x = np.maximum(x, 0.0)
    xp = x ** p
    return xp / (xp + K ** p)

class Model:
    """H, M, A, D on vectors of sites. Parameters may be scalars or arrays of the vector length.
    dH = -H + F(M+u; th0 e^-A);  dM = r(-M + G(H) + gamma D phi(H));  dA = eps(A*(H) - A);
    dD = delta [H - hD]_+ (1 - D);  A* = a H - b H^2.  instant: A = A*(H) at all times; eps = 0: A frozen."""
    def __init__(self, n, q, K, th0, r, a, b=0.0, eps=0.03, instant=False,
                 gamma=0.0, c=1.0, delta=0.0, hD=H_D):
        self.n, self.q, self.K, self.th0, self.r, self.a, self.b = [np.asarray(v, float) for v in (n, q, K, th0, r, a, b)]
        self.eps, self.gamma, self.c, self.delta, self.hD = [np.asarray(v, float) for v in (eps, gamma, c, delta, hD)]
        self.instant = instant

    def target(self, H):
        return self.a * H - self.b * H * H

    def deriv(self, H, M, A, D, u):
        if self.instant:
            A = self.target(H)
        th = self.th0 * np.exp(-A)
        dH = -H + hill(M + u, th, self.n)
        dM = self.r * (-M + hill(H, self.K, self.q) + self.gamma * D * (self.c + (1.0 - self.c) * H))
        dA = np.zeros_like(H) if self.instant else self.eps * (self.target(H) - A)
        dD = self.delta * np.maximum(H - self.hD, 0.0) * (1.0 - D)
        return dH, dM, dA, dD

    def run(self, state, u_fun, t0, t1, monitor=None):
        H, M, A, D = [np.array(x, float) for x in state]
        n = int(round((t1 - t0) / DT)); t = t0
        for _ in range(n):
            u1, u2, u3 = u_fun(t), u_fun(t + DT / 2), u_fun(t + DT)
            k1 = self.deriv(H, M, A, D, u1)
            k2 = self.deriv(H + DT/2*k1[0], M + DT/2*k1[1], A + DT/2*k1[2], D + DT/2*k1[3], u2)
            k3 = self.deriv(H + DT/2*k2[0], M + DT/2*k2[1], A + DT/2*k2[2], D + DT/2*k2[3], u2)
            k4 = self.deriv(H + DT*k3[0], M + DT*k3[1], A + DT*k3[2], D + DT*k3[3], u3)
            H = H + DT/6*(k1[0] + 2*k2[0] + 2*k3[0] + k4[0])
            M = M + DT/6*(k1[1] + 2*k2[1] + 2*k3[1] + k4[1])
            A = A + DT/6*(k1[2] + 2*k2[2] + 2*k3[2] + k4[2])
            D = D + DT/6*(k1[3] + 2*k2[3] + 2*k3[3] + k4[3])
            t = t + DT
            if self.instant:
                A = self.target(H)
            if monitor is not None:
                monitor(t, H, M, A, D)
        return (H, M, A, D), t

def const(u):
    return lambda t: u

def schedule(ub, t_on, dur, U, ka=None):
    """Baseline ub; from t_on for dur either a step to U (ka None) or plaque rising from ub towards U at rate ka."""
    ub, t_on, dur, U = [np.asarray(x, float) for x in (ub, t_on, dur, U)]
    def u(t):
        on = (t >= t_on - 1e-9) & (t < t_on + dur - 1e-9)
        val = U if ka is None else ub + (U - ub) * (1.0 - np.exp(-ka * np.maximum(t - t_on, 0.0)))
        return np.where(on, val, ub)
    return u

TRAPZ = getattr(np, "trapezoid", None) or np.trapz

def rnd(x):  # align times to the step grid
    return np.round(np.asarray(x, float) / DT) * DT


In [ ]:
# ---------------------------------------------------------------- equilibria (scalar parameter sets)
HG = np.linspace(1e-7, 1 - 1e-7, 200001)

def u_curve(p, H=HG, a=None, b=0.0):
    a = p["a"] if a is None else a
    return p["th0"] * np.exp(-(a * H - b * H * H)) * (H / (1 - H)) ** (1 / p["n"]) - hill(H, p["K"], p["q"])

def folds(p, a=None, b=0.0):
    """(H_up, u_up, H_down, u_down): loss of the low state (local max) and of the high state (local min); None if no fold."""
    u = u_curve(p, a=a, b=b); du = np.diff(u)
    idx = np.where(np.sign(du[1:]) != np.sign(du[:-1]))[0] + 1
    if len(idx) < 2: return None
    i_up, i_dn = idx[0], idx[1]
    return HG[i_up], u[i_up], HG[i_dn], u[i_dn]

def eq_H(p, u, branch, a=None, b=0.0):
    """Equilibrium H at constant forcing u on the 'low' or 'high' branch (None if absent)."""
    a = p["a"] if a is None else a
    f = lambda H: u_curve(p, np.array([H]), a=a, b=b)[0] - u
    v = u_curve(p, a=a, b=b) - u
    i = np.where(np.sign(v[1:]) != np.sign(v[:-1]))[0]
    roots = [brentq(f, HG[j], HG[j + 1]) for j in i]
    if not roots: return None
    return roots[0] if branch == "low" else roots[-1]

def eq_state(p, u, branch, D=0.0, gamma=GAMMA, a=None, b=0.0):
    """Equilibrium (H, M, A, D) with fixed D (niche fully persistent): forcing u + gamma D."""
    a = p["a"] if a is None else a
    H = eq_H(p, u + gamma * D, branch, a=a, b=b)
    if H is None: return None
    return H, hill(H, p["K"], p["q"]) + gamma * D, a * H - b * H * H, D

def tile(state, N):
    return tuple(np.full(N, float(x)) for x in state)


In [ ]:
# ---------------------------------------------------------------- therapy protocol and bisection helpers
T_PC = 4.0    # perfect plaque control after debridement (~7 days); H < 0.05 is reached after ~3 units at r = 1
T_PC_SENS = [2.0, 4.0, 10.0, 20.0, 40.0]   # sensitivity of the treated state to this choice (14B5)

def therapy(model, start_state, t_pc=T_PC):
    """Debridement (M -> 0) followed by perfect supragingival plaque control (u = 0) for t_pc.
    Returns state at the end, success flag (H < 0.1) and A retained relative to its pre-therapy value."""
    H, M, A, D = [np.array(x, float) for x in start_state]
    A0 = A.copy()
    M = np.zeros_like(M)
    (H, M, A, D), _ = model.run((H, M, A, D), const(0.0), 0.0, t_pc)
    with np.errstate(invalid="ignore", divide="ignore"):
        retained = np.where(A0 != 0, A / A0, np.nan)
    return (H, M, A, D), H < 0.1, retained

def bisect(converts, lo, hi, n_iter):
    """Vectorized bisection on amplitude. converts(U) -> bool array. Returns critical value (nan if hi does not convert)
    and a flag for cases already converting at lo."""
    lo, hi = np.array(lo, float), np.array(hi, float)
    c_hi = converts(hi); c_lo = converts(lo)
    for _ in range(n_iter):
        mid = 0.5 * (lo + hi)
        c = converts(mid)
        hi = np.where(c, mid, hi); lo = np.where(c, lo, mid)
    crit = np.where(c_hi, hi, np.nan)
    crit = np.where(c_lo, np.asarray(lo, float) * 0 + np.nan, crit)
    return crit, c_lo

def half_life(x, y):
    """First x at which y (starting at 1) falls to 0.5, by linear interpolation."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    for i in range(1, len(y)):
        if y[i] <= 0.5 and y[i - 1] > 0.5:
            return x[i-1] + (y[i-1] - 0.5) * (x[i] - x[i-1]) / (y[i-1] - y[i])
    return np.nan


## 14A — Simulated rechallenge after therapy
Reference set, baselines 0.10 and 0.20. Treated = diseased equilibrium (with D = 0 or residual D = 0.10 > D_irr), debridement and
4 units (~7 days) of perfect plaque control, then baseline. A 21-day experimental-gingivitis challenge (plaque rising from baseline towards U
at rate 0.5, ~3 days) is applied after 0–365 days. Readouts: peak H and area under H during the challenge; whether the site stays
inflamed (H > 0.5) after settling at baseline; whether perfect plaque control then reverses it.
Instant host = memory-free control with identical equilibria (and identical D). D is held fixed (delta = 0).
Single-state set: conditioning at u = 0.4 for 200 units, then natural return to baseline 0.05 (no reset), then the same challenge.


In [ ]:
def run_14A():
    d = outdir("14A_rechallenge"); stamp("14A start")
    D_RES = 0.10
    rec_days = np.array([0, 15, 30, 60, 90, 180, 365], float)
    Ugrid = np.round(np.arange(0.25, 0.7001, 0.05), 3)
    hosts = [("instant", 0.03, True), ("eps=0.03", 0.03, False), ("eps=0.01", 0.01, False)]
    baselines = [0.10, 0.20]
    if FAST:
        rec_days = np.array([0, 30, 180.]); Ugrid = np.array([0.3, 0.4, 0.5]); hosts = hosts[::2]; baselines = [0.20]
    dur = rnd(days(21))
    rows, checks = [], []
    for ub in baselines:
        Ug = np.concatenate([[ub], Ugrid[Ugrid > ub + 1e-9]])          # U = baseline: no-challenge control
        UU, RR = np.meshgrid(Ug, rec_days, indexing="ij"); UU, RR = UU.ravel(), RR.ravel(); N = len(UU)
        for hname, eps, inst in hosts:
            m = Model(**REF, eps=eps, instant=inst, gamma=GAMMA)
            settle = max(10 / eps, 200.0) if not inst else 200.0
            for hist in ["naive", "treated_D0", "treated_Dres"]:
                if hist == "naive":
                    st = tile(eq_state(REF, ub, "low", D=0.0), N); ok = np.ones(N, bool); ret = np.full(N, np.nan)
                else:
                    Dv = 0.0 if hist == "treated_D0" else D_RES
                    st0 = tile(eq_state(REF, ub, "high", D=Dv), N)
                    st, ok, ret = therapy(m, st0)
                t_on = rnd(days(RR))
                u_fun = schedule(ub, t_on, dur, UU, ka=K_ACCESS)
                peak = np.zeros(N); auc = np.zeros(N); A_on = np.full(N, np.nan)
                def mon(t, H, M, A, D):
                    win = (t > t_on + 1e-9) & (t <= t_on + dur + 1e-9)
                    np.maximum(peak, np.where(win, H, 0.0), out=peak)
                    auc[:] += np.where(win, H * DT, 0.0)
                    hit = np.abs(t - t_on - DT) < 1e-6
                    A_on[:] = np.where(hit, A, A_on)
                t_end = float(t_on.max() + dur + settle)
                st2, _ = m.run(st, u_fun, 0.0, t_end, monitor=mon)
                persistent = st2[0] > 0.5
                st3, _ = m.run(st2, const(0.0), 0.0, settle)          # perfect plaque control
                reversible_hyg = st3[0] < 0.5
                stD = (st2[0], np.zeros(N), st2[2], st2[3])                # debridement + plaque control
                st4, _ = m.run(stD, const(0.0), 0.0, settle)
                reversible_deb = st4[0] < 0.5
                for i in range(N):
                    rows.append(dict(set="reference", baseline=ub, host=hname, history=hist, U=UU[i], no_challenge=bool(abs(UU[i] - ub) < 1e-9), recovery_days=RR[i],
                                     therapy_success=bool(ok[i]), A_retained_after_therapy=ret[i], A_at_challenge=A_on[i],
                                     days_since_debridement=RR[i] + (T_PC * UNIT_DAYS if hist != "naive" else np.nan),
                                     peak_H=peak[i], auc_H_days=auc[i] * UNIT_DAYS, persistent=bool(persistent[i]),
                                     reversed_by_plaque_control=bool(reversible_hyg[i]) if persistent[i] else np.nan,
                                     reversed_by_debridement=bool(reversible_deb[i]) if persistent[i] else np.nan))
            stamp(f"14A reference ub={ub} {hname} done")
    ref = pd.DataFrame(rows); ref.to_csv(os.path.join(d, "rechallenge_reference.csv"), index=False)

    # single-state set: dynamic conditioning, natural return, no reset
    rows = []
    ub = 0.05; Ug = np.round(np.arange(0.2, 0.9001, 0.1), 3) if not FAST else np.array([0.3, 0.5])
    rec = rec_days
    UU2, RR2 = np.meshgrid(Ug, rec, indexing="ij"); UU2, RR2 = UU2.ravel(), RR2.ravel(); N2 = len(UU2)
    hosts_s = [("instant", 0.1, True), ("eps=0.1", 0.1, False), ("eps=0.03", 0.03, False), ("eps=0.01", 0.01, False)]
    if FAST: hosts_s = [hosts_s[0], hosts_s[2]]
    for hname, eps, inst in hosts_s:
        m = Model(**SINGLE, eps=eps, instant=inst)
        low = tile(eq_state(SINGLE, ub, "low", gamma=0.0), N2)
        for hist in ["naive", "conditioned"]:
            st = low
            if hist == "conditioned":
                st, _ = m.run(low, const(0.4), 0.0, 200.0)
            t_on = rnd(days(RR2)); u_fun = schedule(ub, t_on, dur, UU2, ka=K_ACCESS)
            peak = np.zeros(N2); auc = np.zeros(N2)
            def mon(t, H, M, A, D):
                win = (t > t_on + 1e-9) & (t <= t_on + dur + 1e-9)
                np.maximum(peak, np.where(win, H, 0.0), out=peak); auc[:] += np.where(win, H * DT, 0.0)
            st2, _ = m.run(st, u_fun, 0.0, float(t_on.max() + dur + 200.0), monitor=mon)
            for i in range(N2):
                rows.append(dict(set="single-state", baseline=ub, host=hname, history=hist, U=UU2[i], recovery_days=RR2[i],
                                 peak_H=peak[i], auc_H_days=auc[i] * UNIT_DAYS, H_after_settling=st2[0][i]))
        stamp(f"14A single-state {hname} done")
    sgl = pd.DataFrame(rows); sgl.to_csv(os.path.join(d, "rechallenge_single_state.csv"), index=False)

    # checks
    nv = ref[ref.history == "naive"]
    checks.append(("naive response independent of recovery time (max spread of peak H)",
                   float(nv.groupby(["baseline", "host", "U"]).peak_H.agg(np.ptp).max()), None))
    checks.append(("therapy succeeded in every treated case", bool(ref[ref.history != "naive"].therapy_success.all()),
                   bool(ref[ref.history != "naive"].therapy_success.all())))
    ins = ref[(ref.host == "instant") & (ref.history == "treated_D0") & (ref.recovery_days >= 15)].merge(
        ref[(ref.host == "instant") & (ref.history == "naive")], on=["baseline", "U", "recovery_days"])
    checks.append(("instant host: treated (D=0) peak H equals naive from 15 days (max abs diff)",
                   float((ins.peak_H_x - ins.peak_H_y).abs().max()), bool((ins.peak_H_x - ins.peak_H_y).abs().max() < 1e-4)))
    checks.append(("single-state set: H after settling below 0.5 in every run (monostable)",
                   float(sgl.H_after_settling.max()), bool(sgl.H_after_settling.max() < 0.5)))
    save_checks(d, checks)

    # summary tables
    piv = ref[ref.U.round(3) == 0.4].pivot_table(index=["baseline", "host", "history"], columns="recovery_days", values="peak_H")
    piv.to_csv(os.path.join(d, "summary_peakH_U040.csv"))
    ref[ref.no_challenge].pivot_table(index=["baseline", "host", "history"], columns="recovery_days", values="persistent",
                                      aggfunc="max").to_csv(os.path.join(d, "summary_relapse_without_challenge.csv"))
    pers = ref[ref.persistent & ~ref.no_challenge].groupby(["baseline", "host", "history", "recovery_days"]).U.min().rename("lowest_U_persistent")
    pers.to_csv(os.path.join(d, "summary_lowest_persistent_U.csv"))

    # figures
    fig, axes = plt.subplots(len(baselines), 3, figsize=(13, 4 * len(baselines)), squeeze=False)
    Usel = 0.4 if 0.4 in set(np.round(Ugrid, 3)) else Ugrid[len(Ugrid) // 2]
    for i, ub in enumerate(baselines):
        for j, hist in enumerate(["naive", "treated_D0", "treated_Dres"]):
            ax = axes[i, j]
            for hname, _, _ in hosts:
                g = ref[(ref.baseline == ub) & (ref.host == hname) & (ref.history == hist) & (ref.U.round(3) == round(Usel, 3))]
                ax.plot(g.recovery_days, g.peak_H, "o-", label=hname)
                ax.scatter(g.recovery_days[g.persistent], g.peak_H[g.persistent], s=90, facecolors="none", edgecolors="k")
            ax.set_title(f"baseline {ub}, {hist}, U = {Usel}"); ax.set_xlabel("days since therapy"); ax.set_ylabel("peak H")
            ax.set_ylim(0, 1.02); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(d, "fig14A_01_peakH_vs_time.png"), dpi=200); plt.close(fig)
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    for (hname, hist), g in sgl[sgl.U.round(3) == 0.3].groupby(["host", "history"]):
        ax.plot(g.recovery_days, g.peak_H, "o-" if hist == "conditioned" else "x--", label=f"{hname}, {hist}")
    ax.set_xlabel("days after return to baseline"); ax.set_ylabel("peak H (U = 0.3)"); ax.set_title("single-state set")
    ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(os.path.join(d, "fig14A_02_single_state.png"), dpi=200); plt.close(fig)

    save_json(d, "parameters.json", dict(reference=REF, single_state=SINGLE, baselines=baselines, D_res=D_RES, gamma=GAMMA,
              therapy="M -> 0, then u = 0 for %g units" % T_PC, challenge="21 days, plaque rising at k = %g" % K_ACCESS,
              U_grid=list(Ugrid), recovery_days=list(rec_days), unit_days=UNIT_DAYS, dt=DT))
    with open(os.path.join(d, "README.txt"), "w") as f:
        f.write("14A - Simulated experimental-gingivitis rechallenge after therapy\n"
                "rechallenge_reference.csv: one row per baseline x host x history x U x recovery time.\n"
                "  peak_H, auc_H_days during the 21-day challenge; persistent = H > 0.5 after settling at baseline;\n"
                "  reversed_by_plaque_control / reversed_by_debridement: outcome of u = 0 (and M -> 0) for the settling time.\n"
                "  histories: naive (D = 0); treated_D0; treated_Dres (D = 0.10, above D_irr). Rows with no_challenge = True (U = baseline) show relapse without challenge. D fixed throughout.\n"
                "rechallenge_single_state.csv: single-state set, conditioning u = 0.4 for 200 units, natural return to 0.05.\n"
                "summary_peakH_U040.csv, summary_lowest_persistent_U.csv, summary_relapse_without_challenge.csv, figures fig14A_*.\n")
    save_versions(d); stamp("14A done")


## 14B — Post-therapy window from a model-defined therapy
Reference set, baseline 0.20, D = 0. Histories: naive; resolved gingivitis (u = 0.45 for 5/ε); treated_protocol (diseased equilibrium,
debridement, 4 units of plaque control, baseline); treated_reset (Experiment 08 definition, for comparison).
B1 critical challenge vs duration; B2 vulnerability vs recovery time at duration 2/ε, half-lives, window lengths, decay of A;
B3 the same at r = 0.03; B4 relapse ceiling without challenge; B5 sensitivity to the length of plaque control.


In [ ]:
def init_history(m, p, ub, hist, eps, N, inst, t_pc=None):
    low = eq_state(p, ub, "low", gamma=0.0)
    if hist == "naive":
        return tile(low, N)
    if hist == "gingivitis":
        st, _ = m.run(tile(low, N), const(0.45), 0.0, 5 / eps if not inst else 100.0)
        return st
    high = eq_state(p, ub, "high", gamma=0.0)
    if hist == "treated_reset":
        return tile((low[0], low[1], high[2], 0.0), N)
    if hist == "treated_protocol":
        st, ok, _ = therapy(m, tile(high, N), T_PC if t_pc is None else t_pc)
        assert ok.all(), "therapy failed"
        return st
    raise ValueError(hist)

def critical_challenge(m, states, ub, t_on, dur, settle, hi=4.0, n_iter=16):
    t_on, dur = rnd(t_on), rnd(dur)
    t_end = float((t_on + dur).max() + settle)
    def converts(U):
        st, _ = m.run(states, schedule(ub, t_on, dur, U), 0.0, t_end)
        return st[0] > 0.5
    N = len(t_on)
    return bisect(converts, np.full(N, ub), np.full(N, hi), n_iter)

def run_14B():
    d = outdir("14B_tipping_window"); stamp("14B start")
    ub = 0.20; checks = []
    hosts = [("instant", 0.1, True), ("eps=0.1", 0.1, False), ("eps=0.03", 0.03, False), ("eps=0.01", 0.01, False)]
    durs = [5.0, 10.0, 20.0, 50.0, 100.0, 200.0, 500.0]
    eT = [0.0, 0.1, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]
    n_iter = 16
    if FAST:
        hosts = [hosts[0], hosts[2]]; durs = [20.0, 100.0]; eT = [0.0, 1.0, 4.0]; n_iter = 8
    hists = ["naive", "gingivitis", "treated_protocol", "treated_reset"]

    # B1: duration
    rows = []
    for hname, eps, inst in hosts:
        m = Model(**REF, eps=eps, instant=inst)
        dl = sorted(set(durs + ([2 / eps] if not inst else [20.0])))
        settle = 10 / eps if not inst else 200.0
        states, lab = [], []
        for h in hists:
            s = init_history(m, REF, ub, h, eps, len(dl), inst); states.append(s); lab += [(h, x) for x in dl]
        st = tuple(np.concatenate([s[k] for s in states]) for k in range(4))
        dd = np.array([x for _, x in lab])
        crit, relapse = critical_challenge(m, st, ub, np.zeros(len(dd)), dd, settle, n_iter=n_iter)
        for (h, x), c, rl in zip(lab, crit, relapse):
            rows.append(dict(host=hname, history=h, duration=x, duration_days=x * UNIT_DAYS, critical=c, relapse_without_challenge=rl))
        stamp(f"14B1 {hname}")
    b1 = pd.DataFrame(rows)
    nv = b1[b1.history == "naive"][["host", "duration", "critical"]].rename(columns={"critical": "critical_naive"})
    b1 = b1.merge(nv, on=["host", "duration"])
    b1["relative_reduction_vs_naive"] = 1 - b1.critical / b1.critical_naive
    inst_ref = b1[(b1.host == "instant") & (b1.history == "naive")][["duration", "critical"]].rename(columns={"critical": "critical_instant"})
    b1 = b1.merge(inst_ref, on="duration", how="left")
    b1["relative_reduction_vs_instant"] = 1 - b1.critical / b1.critical_instant
    b1.to_csv(os.path.join(d, "B1_critical_vs_duration.csv"), index=False)

    # reproduction of Experiment 08
    def get(h, hist, dur):
        g = b1[(b1.host == h) & (b1.history == hist) & (np.isclose(b1.duration, dur, atol=0.05))]
        return float(g.critical.iloc[0]) if len(g) else np.nan
    if not FAST:
        for h, hist, dur, ref_v in [("eps=0.03", "treated_reset", 2 / 0.03, 0.2728271484375), ("eps=0.03", "naive", 2 / 0.03, 0.5650634765625),
                                    ("eps=0.01", "treated_reset", 200.0, 0.2542724609375), ("instant", "naive", 20.0, 0.47600097656249996)]:
            v = get(h, hist, dur)
            checks.append((f"reproduces Exp 08 {h} {hist} duration {dur:.1f}: {v:.4f} vs {ref_v:.4f}", v - ref_v, bool(abs(v - ref_v) < 0.003)))

    # B2: recovery time at duration 2/eps
    rows, decay_rows = [], []
    for hname, eps, inst in hosts:
        m = Model(**REF, eps=eps, instant=inst)
        dch = 2 / eps if not inst else 20.0
        settle = 10 / eps if not inst else 200.0
        trec = np.array(eT) / eps if not inst else np.array([0, 0.5, 1, 2, 5, 10, 20, 50.0])
        states, lab = [], []
        for h in ["gingivitis", "treated_protocol", "treated_reset"]:
            s = init_history(m, REF, ub, h, eps, len(trec), inst); states.append(s); lab += [(h, x) for x in trec]
        states.append(init_history(m, REF, ub, "naive", eps, 1, inst)); lab.append(("naive", 0.0))
        st = tuple(np.concatenate([s[k] for s in states]) for k in range(4))
        t_on = np.array([x for _, x in lab])
        crit, _ = critical_challenge(m, st, ub, t_on, np.full(len(t_on), dch), settle, n_iter=n_iter)
        un = crit[-1]
        for (h, x), c in zip(lab[:-1], crit[:-1]):
            rows.append(dict(host=hname, history=h, T_rec=x, eps_T_rec=x * eps if not inst else np.nan,
                             T_rec_days=x * UNIT_DAYS, duration=dch, critical=c, critical_naive=un))
        if not inst:   # decay of A after treated_protocol, no challenge
            s0 = init_history(m, REF, ub, "treated_protocol", eps, 1, inst); A0 = s0[2][0]
            tt = np.linspace(0, 8 / eps, 161); At = []
            s = s0; tprev = 0.0
            for tk in tt:
                s, _ = m.run(s, const(ub), 0.0, rnd(tk - tprev)); tprev = tprev + rnd(tk - tprev); At.append(s[2][0])
            for tk, a in zip(tt, At): decay_rows.append(dict(host=hname, eps_T=tk * eps, A_rel=a / A0))
        stamp(f"14B2 {hname}")
    b2 = pd.DataFrame(rows)
    b2["ratio_to_naive"] = b2.critical / b2.critical_naive
    out = []
    for (h, hist), g in b2.groupby(["host", "history"]):
        g = g.sort_values("T_rec"); c0 = g.critical.iloc[0]; un = g.critical_naive.iloc[0]
        V = (un - g.critical) / (un - c0) if un != c0 else g.critical * np.nan
        b2.loc[g.index, "V"] = V.values
        hl = half_life(g.eps_T_rec, V) if h != "instant" else np.nan
        wl = {}
        for f in [0.5, 0.6, 0.7, 0.8, 0.9]:
            ok = g.critical.values >= f * un - 1e-12
            if ok[0]: wl[f] = 0.0
            elif ok.any():
                j = np.argmax(ok); x0, x1 = g.eps_T_rec.values[j - 1], g.eps_T_rec.values[j]
                y0, y1 = g.critical.values[j - 1], g.critical.values[j]
                wl[f] = x0 + (f * un - y0) * (x1 - x0) / (y1 - y0)
            else: wl[f] = np.nan
        out.append(dict(host=h, history=hist, V_half_life_memory_units=hl, **{f"window_at_{f}_naive_memory_units": wl[f] for f in wl}))
    b2.to_csv(os.path.join(d, "B2_window_vs_recovery.csv"), index=False)
    hl = pd.DataFrame(out)
    dec = pd.DataFrame(decay_rows); dec.to_csv(os.path.join(d, "B2_memory_decay_treated_protocol.csv"), index=False)
    for h, g in dec.groupby("host"):
        hl = pd.concat([hl, pd.DataFrame([dict(host=h, history="A after treated_protocol", V_half_life_memory_units=half_life(g.eps_T, g.A_rel))])])
    hl["passive_ln2"] = np.log(2)
    hl.to_csv(os.path.join(d, "B2_half_lives_and_window_lengths.csv"), index=False)

    # B3: r = 0.03
    rows = []
    for r in [1.0, 0.03]:
        p = dict(REF, r=r)
        for hname, eps, inst in [h for h in hosts if h[0] in ("instant", "eps=0.03", "eps=0.01")]:
            m = Model(**p, eps=eps, instant=inst)
            dch = 2 / eps if not inst else 20.0; settle = (10 / eps if not inst else 200.0) + 10 / r
            states = [init_history(m, p, ub, "naive", eps, 1, inst), init_history(m, p, ub, "treated_protocol", eps, 1, inst)]
            st = tuple(np.concatenate([s[k] for s in states]) for k in range(4))
            crit, _ = critical_challenge(m, st, ub, np.zeros(2), np.full(2, dch), settle, n_iter=n_iter)
            rows.append(dict(r=r, host=hname, duration=dch, critical_naive=crit[0], critical_treated=crit[1], ratio=crit[1] / crit[0]))
    pd.DataFrame(rows).to_csv(os.path.join(d, "B3_recolonization_check.csv"), index=False); stamp("14B3")

    # B5: sensitivity of the treated state to the length of post-debridement plaque control
    rows = []
    for hname, eps, inst in [h for h in hosts if not h[2]]:
        m = Model(**REF, eps=eps)
        high = eq_state(REF, ub, "high", gamma=0.0)
        states, Aret = [], []
        for tp in T_PC_SENS:
            s, ok, ret = therapy(m, tile(high, 1), tp); states.append(s); Aret.append((float(ret[0]), bool(ok[0])))
        st = tuple(np.concatenate([s[k] for s in states]) for k in range(4))
        crit, _ = critical_challenge(m, st, ub, np.zeros(len(T_PC_SENS)), np.full(len(T_PC_SENS), 2 / eps), 10 / eps, n_iter=n_iter)
        for tp, (ar, ok), c in zip(T_PC_SENS, Aret, crit):
            rows.append(dict(host=hname, plaque_control_units=tp, plaque_control_days=tp * UNIT_DAYS, therapy_success=ok,
                             A_retained=ar, critical_treated=c))
    pd.DataFrame(rows).to_csv(os.path.join(d, "B5_plaque_control_sensitivity.csv"), index=False); stamp("14B5")

    # B4: relapse ceiling (no challenge); therapy at the same baseline
    rows = []
    U_STATIC = folds(REF)[1]
    ubs = np.round(np.arange(0.07, 0.4651, 0.005), 3) if not FAST else np.round(np.arange(0.20, 0.40, 0.04), 3)
    hl_hosts = [("instant", 0.1, True)] + [(f"eps={e}", e, False) for e in ([0.3, 0.1, 0.03, 0.01, 0.003] if not FAST else [0.03])]
    for hname, eps, inst in hl_hosts:
        m = Model(**REF, eps=eps, instant=inst); settle = 10 / eps if not inst else 200.0
        for kind in ["treated_protocol", "treated_reset"]:
            H = []; M = []; A = []; okv = []
            for u in ubs:
                hi_, lo_ = eq_state(REF, u, "high", gamma=0.0), eq_state(REF, u, "low", gamma=0.0)
                if kind == "treated_reset":
                    s = (lo_[0], lo_[1], hi_[2], 0.0); ok = True
                else:
                    s, ok, _ = therapy(m, tile(hi_, 1)); s = tuple(x[0] for x in s); ok = bool(ok[0])
                H.append(s[0]); M.append(s[1]); A.append(s[2]); okv.append(ok)
            st = (np.array(H), np.array(M), np.array(A), np.zeros(len(ubs)))
            st2, _ = m.run(st, const(ubs), 0.0, settle)
            rel = st2[0] > 0.5
            ceil = np.nan
            for i in range(len(ubs)):
                if rel[i:].all(): ceil = ubs[i]; break
            for u, rl, ok in zip(ubs, rel, okv):
                rows.append(dict(host=hname, kind=kind, baseline=u, relapse=bool(rl), therapy_success=ok, ceiling=ceil,
                                 ceiling_over_static=ceil / U_STATIC))
        stamp(f"14B4 {hname}")
    b4 = pd.DataFrame(rows); b4.to_csv(os.path.join(d, "B4_relapse_ceiling.csv"), index=False)
    b4.groupby(["host", "kind"]).ceiling.first().to_csv(os.path.join(d, "B4_relapse_ceiling_summary.csv"))

    # figures
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for (h, hist), g in b1.groupby(["host", "history"]):
        axes[0].plot(g.duration, g.critical, "o-" if h != "instant" else "k--", label=f"{h} {hist}", ms=3)
    axes[0].set_xscale("log"); axes[0].set_xlabel("challenge duration"); axes[0].set_ylabel("critical challenge"); axes[0].legend(fontsize=6)
    for (h, hist), g in b2[b2.history != "gingivitis"].groupby(["host", "history"]):
        if h != "instant": axes[1].plot(g.eps_T_rec, g.V, "o-", label=f"{h} {hist}", ms=3)
    x = np.linspace(0, 8, 100); axes[1].plot(x, np.exp(-x), "k:", label="e^-εT")
    axes[1].set_xlabel("εT_rec"); axes[1].set_ylabel("V"); axes[1].legend(fontsize=6)
    fig.tight_layout(); fig.savefig(os.path.join(d, "fig14B_01_window.png"), dpi=200); plt.close(fig)

    save_checks(d, checks)
    save_json(d, "parameters.json", dict(reference=REF, baseline=ub, durations=durs, eps_T_rec=eT, therapy_plaque_control=T_PC,
              gingivitis="u = 0.45 for 5/eps", settle="10/eps (instant 200)", bisection_iterations=n_iter, dt=DT))
    with open(os.path.join(d, "README.txt"), "w") as f:
        f.write("14B - Post-therapy window with a model-defined therapy\n"
                "B1_critical_vs_duration.csv: critical step challenge vs duration (recovery 0), reductions vs naive and vs instant host.\n"
                "B2_window_vs_recovery.csv: critical challenge and V vs recovery (duration 2/eps); B2_half_lives_and_window_lengths.csv;\n"
                "  B2_memory_decay_treated_protocol.csv: A after therapy relative to its pre-therapy value.\n"
                "B3_recolonization_check.csv: naive/treated at r = 1 and r = 0.03.\n"
                "B4_relapse_ceiling.csv (+ summary): lowest baseline above which treated sites relapse without challenge.\n"
                "B5_plaque_control_sensitivity.csv: treated critical challenge vs length of post-debridement plaque control.\n"
                "treated_protocol = diseased equilibrium at the baseline, M -> 0, u = 0 for T_PC = 4 units (~7 days), then baseline.\n"
                "treated_reset = Experiment 08 definition (H, M healthy; A of the diseased state).\n")
    save_versions(d); stamp("14B done")


## 14C — Ratchet: conversion, first inflamed recall, first damaging recall; latency vs passage time
Same grid as Experiment 07. For each cycle: entrapped (H > 0.5 throughout), inflamed at recall (H > 0.5 at the end), damaging at recall
(H > 0.8 at the end). The reduced map Θ(A) is recomputed at the latency peaks (T = 60) and the passage time is compared with
N(1 − e^−εT) and with NεT. Thresholds at T = 60 are repeated at r = 0.03.


In [ ]:
EXP07_PC = {0.03: 0.5691503940585378, 0.01: 0.5702185679766016, 0.003: 0.5702696764335574}   # latency_scaling_fits.csv
EXP07_T60 = {"instant": 0.50, "eps=0.3": 0.53, "eps=0.1": 0.57, "eps=0.03": 0.57, "eps=0.01": 0.58, "eps=0.003": 0.58}

def ratchet(p, eps, inst, T, peaks, ncyc):
    peaks = np.asarray(peaks, float); N = len(peaks)
    umax = peaks / (1 - np.exp(-K_REGROW * T))
    m = Model(**p, eps=eps, instant=inst)
    st = tuple(np.zeros(N) for _ in range(4))
    u_fun = lambda t: umax * (1 - np.exp(-K_REGROW * t))
    ent = np.zeros((ncyc, N), bool); hrec = np.zeros((ncyc, N))
    for c in range(ncyc):
        hmin = np.full(N, 9.0)
        def mon(t, H, M, A, D): np.minimum(hmin, H, out=hmin)
        st, _ = m.run(st, u_fun, 0.0, T, monitor=mon)
        ent[c] = hmin > 0.5; hrec[c] = st[0]
    # conversion cycle: first c from which all later cycles are entrapped (requires the last cycle entrapped)
    conv = np.full(N, np.nan)
    tail = np.flip(np.logical_and.accumulate(np.flip(ent, 0), 0), 0)
    for i in range(N):
        if tail[-1, i]:
            conv[i] = np.argmax(tail[:, i]) + 1
    def first(mask):
        out = np.full(N, np.nan)
        for i in range(N):
            if mask[:, i].any(): out[i] = np.argmax(mask[:, i]) + 1
        return out
    return conv, first(hrec > 0.5), first(hrec > 0.8)

def theta_map(p, T, peak, A_grid, ncyc=30):
    """Mean H over one interval (last of ncyc) with memory frozen at A, from a naive start; entrapped flag."""
    N = len(A_grid); umax = peak / (1 - np.exp(-K_REGROW * T))
    m = Model(**p, eps=0.0)
    st = (np.zeros(N), np.zeros(N), np.array(A_grid, float), np.zeros(N))
    u_fun = lambda t: umax * (1 - np.exp(-K_REGROW * t))
    for c in range(ncyc):
        acc = np.zeros(N); hmin = np.full(N, 9.0)
        def mon(t, H, M, A, D):
            acc[:] += H * DT; np.minimum(hmin, H, out=hmin)
        st, _ = m.run(st, u_fun, 0.0, T, monitor=mon)
    return acc / T, hmin > 0.5

def run_14C():
    d = outdir("14C_ratchet"); stamp("14C start"); checks = []
    Ts = [10.0, 15.0, 20.0, 30.0, 40.0, 60.0, 80.0, 120.0, 160.0]
    peaks = np.round(np.arange(0.30, 0.9001, 0.01), 3)
    hosts = [("instant", 0.1, True)] + [(f"eps={e}", e, False) for e in [0.3, 0.1, 0.03, 0.01, 0.003]]
    if FAST:
        Ts = [60.0]; peaks = np.round(np.arange(0.45, 0.65, 0.01), 3); hosts = [hosts[0], hosts[4]]
    rows = []
    for hname, eps, inst in hosts:
        for T in Ts:
            ncyc = 20 if inst else int(max(40, np.ceil(40 / (eps * T))))
            if FAST and not inst: ncyc = min(ncyc, 120)
            conv, finf, fdam = ratchet(REF, eps, inst, T, peaks, ncyc)
            for i, pk in enumerate(peaks):
                rows.append(dict(host=hname, T=T, kT=K_REGROW * T, peak=pk, cycles_followed=ncyc, conversion_cycle=conv[i],
                                 first_inflamed_recall=finf[i], first_damaging_recall=fdam[i]))
        stamp(f"14C grid {hname}")
    g = pd.DataFrame(rows); g.to_csv(os.path.join(d, "C1_grid.csv"), index=False)
    thr = []
    for (h, T), gg in g.groupby(["host", "T"]):
        def lowest(col):
            ok = gg[col].notna().values
            for i in range(len(ok)):
                if ok[i:].all(): return gg.peak.values[i]
            return np.nan
        thr.append(dict(host=h, T=T, kT=K_REGROW * T, threshold_conversion=lowest("conversion_cycle"),
                        threshold_inflamed_recall=lowest("first_inflamed_recall"), threshold_damaging_recall=lowest("first_damaging_recall")))
    thr = pd.DataFrame(thr)
    inst_thr = thr[thr.host == "instant"][["T", "threshold_conversion"]].rename(columns={"threshold_conversion": "instant_threshold"})
    thr = thr.merge(inst_thr, on="T", how="left")
    thr["static_fold"] = folds(REF)[1]
    thr["over_static"] = thr.threshold_conversion / thr.static_fold
    thr["over_instant"] = thr.threshold_conversion / thr.instant_threshold
    thr.to_csv(os.path.join(d, "C1_thresholds.csv"), index=False)
    for h, ref_v in EXP07_T60.items():
        r_ = thr[(thr.host == h) & (thr["T"] == 60.0)]
        if len(r_):
            v = float(r_.threshold_conversion.iloc[0])
            checks.append((f"T=60 threshold {h} reproduces Exp 07 ({v} vs {ref_v})", v - ref_v, bool(abs(v - ref_v) < 0.0101)))

    # C2 latency vs passage time at T = 60
    T = 60.0; rows = []
    x = np.logspace(np.log10(2e-4), np.log10(0.28), 16 if not FAST else 3)
    Agrid = np.round(np.arange(0.0, 1.2001, 0.002 if not FAST else 0.02), 4)
    for eps, pc in (EXP07_PC.items() if not FAST else [(0.01, EXP07_PC[0.01])]):
        lp = pc * (1 + x)
        conv, _, _ = ratchet(REF, eps, False, T, lp, 600 if not FAST else 100)
        for pk, N in zip(lp, conv):
            Th, ent = theta_map(REF, T, pk, Agrid, ncyc=30 if not FAST else 10)
            Ac = Agrid[np.argmax(ent)] if ent.any() else Agrid[-1]
            sel = Agrid <= Ac
            gap = REF["a"] * Th[sel] - Agrid[sel]
            s = TRAPZ(1 / gap, Agrid[sel]) if (gap > 0).all() else np.inf
            lam = np.exp(-eps * T)
            rows.append(dict(eps=eps, peak=pk, distance=pk / pc - 1, cycles=N, A_c=Ac, passage_s=s,
                             ratio_N_epsT=N * eps * T / s, ratio_N_1mlam=N * (1 - lam) / s))
        stamp(f"14C latency eps={eps}")
    lat = pd.DataFrame(rows); lat.to_csv(os.path.join(d, "C2_latency_vs_passage.csv"), index=False)
    fits = []
    for eps, gl in lat.groupby("eps"):
        gl = gl[gl.cycles.notna() & (gl.distance < 0.02)]
        if len(gl) > 3:
            b = np.polyfit(np.log(gl.distance), np.log(gl.cycles), 1)[0]
            fits.append(dict(eps=eps, exponent_near_threshold=-b, n_points=len(gl)))
    pd.DataFrame(fits).to_csv(os.path.join(d, "C2_exponent_near_threshold.csv"), index=False)

    # C3 r = 0.03 at T = 60
    rows = []
    pk = np.round(np.arange(0.45, 0.7001, 0.01), 3)
    for r in [1.0, 0.03]:
        p = dict(REF, r=r)
        for hname, eps, inst in [("instant", 0.1, True), ("eps=0.03", 0.03, False), ("eps=0.01", 0.01, False)]:
            if FAST and hname == "eps=0.03": continue
            ncyc = 20 if inst else int(max(40, np.ceil(40 / (eps * 60))))
            conv, _, _ = ratchet(p, eps, inst, 60.0, pk, ncyc)
            ok = ~np.isnan(conv); t_ = np.nan
            for i in range(len(ok)):
                if ok[i:].all(): t_ = pk[i]; break
            rows.append(dict(r=r, host=hname, threshold_T60=t_))
    pd.DataFrame(rows).to_csv(os.path.join(d, "C3_recolonization_check.csv"), index=False)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for h, gg in thr.groupby("host"):
        ax[0].plot(gg.kT, gg.over_instant, "o-", label=h, ms=3)
    ax[0].axhline(1, color="k", lw=0.8); ax[0].set_xlabel("kT"); ax[0].set_ylabel("threshold / instant host"); ax[0].legend(fontsize=7)
    for eps, gl in lat.groupby("eps"):
        ax[1].semilogx(gl.distance, gl.ratio_N_1mlam, "o-", label=f"eps={eps}, N(1-λ)/s", ms=3)
        ax[1].semilogx(gl.distance, gl.ratio_N_epsT, "x--", label=f"eps={eps}, NεT/s", ms=3)
    ax[1].axhline(1, color="k", lw=0.8); ax[1].set_xlabel("distance above threshold"); ax[1].legend(fontsize=6)
    fig.tight_layout(); fig.savefig(os.path.join(d, "fig14C_01_thresholds_latency.png"), dpi=200); plt.close(fig)
    save_checks(d, checks)
    save_json(d, "parameters.json", dict(reference=REF, T_list=Ts, peaks=[float(peaks[0]), float(peaks[-1]), 0.01], k=K_REGROW,
              follow_up="instant 20 cycles; memory max(40, 40/(eps T)) cycles", latency_pc_from_Exp07=EXP07_PC, dt=DT))
    with open(os.path.join(d, "README.txt"), "w") as f:
        f.write("14C - Ratchet with alternative criteria; latency vs reduced-map passage time\n"
                "C1_grid.csv: per host, T, peak: conversion cycle (full entrapment), first recall with H > 0.5, first recall with H > 0.8.\n"
                "C1_thresholds.csv: thresholds by each criterion; ratios to the static fold and to the instant host.\n"
                "C2_latency_vs_passage.csv: cycles to conversion vs passage time s = int dA/(aTheta - A); ratios N eps T/s and N(1-e^-eps T)/s.\n"
                "C2_exponent_near_threshold.csv: log-log slope within 2% of threshold.\n"
                "C3_recolonization_check.csv: T = 60 thresholds at r = 1 and r = 0.03.\n")
    save_versions(d); stamp("14C done")


## 14D — Episodic activity with activity-independent measures
Reference set with structural memory (γ = 0.8, δ = 0.001, hD = 0.8), T = 60, treatment on detection (M → 0 if H > 0.5 at recall),
100 sites × 150 intervals. Part 1 reproduces the eight Experiment 12 conditions (exactly, if its peak_sequences.npz is available).
Part 2 scans the mean peak with common standard-normal draws and then reruns each host at the mean that gives active fractions of
0.10, 0.20, 0.30 and 0.50, so hosts are compared at equal activity.


In [ ]:
def episodic(eps, inst, peaks, therapy_on=True, T=60.0):
    peaks = np.asarray(peaks, float); N, I = peaks.shape
    m = Model(**REF, eps=eps, instant=inst, gamma=GAMMA, delta=0.001, hD=H_D)
    st = tuple(np.zeros(N) for _ in range(4))
    loss = np.zeros((N, I)); treated = np.zeros((N, I), bool)
    for i in range(I):
        umax = peaks[:, i] / (1 - np.exp(-K_REGROW * T))
        D0 = st[3].copy()
        st, _ = m.run(st, lambda t: umax * (1 - np.exp(-K_REGROW * t)), 0.0, T)
        loss[:, i] = st[3] - D0
        if therapy_on:
            tr = st[0] > 0.5; treated[:, i] = tr
            st = (st[0], np.where(tr, 0.0, st[1]), st[2], st[3])
    return loss, treated, st[3]

def runs(x, val):
    out, c = [], 0
    for v in x:
        if v == val: c += 1
        elif c: out.append(c); c = 0
    if c: out.append(c)
    return out

def site_metrics(loss):
    act = loss > 1e-4; N, I = act.shape; res = []
    for s in range(N):
        a = act[s].astype(int); l = loss[s]; r = {}
        r["active_fraction"] = a.mean()
        if 0 < a.sum() < I:
            xm = a.mean(); r["lag1_autocorr"] = np.sum((a[:-1] - xm) * (a[1:] - xm)) / np.sum((a - xm) ** 2)
            prev, nxt = a[:-1], a[1:]
            pa = nxt[prev == 1].mean() if (prev == 1).any() else np.nan
            pq = nxt[prev == 0].mean() if (prev == 0).any() else np.nan
            r["P_act_given_act_minus_P_act_given_quiet"] = pa - pq
        else:
            r["lag1_autocorr"] = np.nan; r["P_act_given_act_minus_P_act_given_quiet"] = np.nan
        qr, ar = runs(a, 0), runs(a, 1)
        r["mean_quiet_run"] = np.mean(qr) if qr else np.nan
        r["mean_active_run"] = np.mean(ar) if ar else np.nan
        r["burstiness_quiet_runs"] = (np.std(qr) - np.mean(qr)) / (np.std(qr) + np.mean(qr)) if len(qr) >= 2 else np.nan
        third = I // 3
        q1, q3 = runs(a[:third], 0), runs(a[-third:], 0)
        r["quiet_run_first_third"] = np.mean(q1) if q1 else np.nan
        r["quiet_run_last_third"] = np.mean(q3) if q3 else np.nan
        tot = l.sum()
        k = max(1, int(round(0.1 * I)))
        r["top10_share"] = np.sort(l)[::-1][:k].sum() / tot if tot > 0 else np.nan
        la = np.sort(l[a == 1])
        if len(la) >= 2 and la.sum() > 0:
            n_ = len(la); r["gini_loss_active_intervals"] = (2 * np.sum(np.arange(1, n_ + 1) * la) / (n_ * la.sum())) - (n_ + 1) / n_
        else:
            r["gini_loss_active_intervals"] = np.nan
        r["D_final_site"] = np.nan
        res.append(r)
    df = pd.DataFrame(res)
    return df.mean(numeric_only=True), df

def run_14D():
    d = outdir("14D_episodic"); stamp("14D start"); checks = []
    hosts = [("instant", 0.1, True), ("eps=0.03", 0.03, False), ("eps=0.01", 0.01, False)]
    I = 150; NS = 100
    if FAST: I, NS = 40, 20; hosts = [hosts[0], hosts[2]]
    npz = None
    for pth in glob.glob("**/peak_sequences.npz", recursive=True):
        npz = np.load(pth); print("using", pth); break
    if npz is None:
        for zp in glob.glob("**/Experiment_12_Episodic_Activity_EXPORT*.zip", recursive=True):
            with zipfile.ZipFile(zp) as z:
                for nm in z.namelist():
                    if nm.endswith("peak_sequences.npz"):
                        z.extract(nm, "exp12"); npz = np.load(os.path.join("exp12", nm)); print("using", zp); break
            if npz is not None: break
    rng = np.random.default_rng(20261015)
    def lognorm(mean, cv, Z):
        s2 = np.log(1 + cv ** 2); return np.exp(np.log(mean) - s2 / 2 + np.sqrt(s2) * Z)
    if npz is not None:
        P, Mn, Cv = npz["peaks"], npz["mean"], npz["cv"]
        conds = sorted(set(zip(Mn.round(3), Cv.round(3))))
        seq = {c: P[(Mn.round(3) == c[0]) & (Cv.round(3) == c[1])][:NS, :I] for c in conds}
        sel = (Mn.round(3) == 0.5) & (Cv.round(3) == 0.3)
        s2 = np.log(1 + 0.3 ** 2); Z = (np.log(P[sel][:NS, :I]) - (np.log(0.5) - s2 / 2)) / np.sqrt(s2)
    else:
        conds = [(m_, c_) for m_ in [0.45, 0.5, 0.55, 0.6] for c_ in [0.15, 0.3]]
        seq = {c: lognorm(c[0], c[1], rng.standard_normal((NS, I))) for c in conds}
        Z = rng.standard_normal((NS, I))
    if FAST: conds = [(0.5, 0.3)]

    def evaluate(tag, hname, eps, inst, mean, cv, peaks, store):
        loss, treated, Df = episodic(eps, inst, peaks)
        mm, _ = site_metrics(loss)
        row = dict(part=tag, host=hname, mean_peak=mean, cv=cv, debridements_per_site=treated.sum(1).mean(), D_final=Df.mean(), **mm.to_dict())
        row.pop("D_final_site", None); store.append(row)
        return row, loss

    rows = []; per_site = {}
    for hname, eps, inst in hosts:
        for c in conds:
            row, loss = evaluate("original", hname, eps, inst, c[0], c[1], seq[c], rows)
            per_site[f"original_{hname}_m{c[0]}_cv{c[1]}"] = loss
        stamp(f"14D original {hname}")
    orig = pd.DataFrame(rows)
    if npz is not None and not FAST:
        r_ = orig[(orig.host == "eps=0.01") & (orig.mean_peak.round(3) == 0.5) & (orig.cv.round(3) == 0.3)].iloc[0]
        checks.append(("reproduces Exp 12 lag-1 (eps=0.01, 0.5, 0.3) 0.5163", r_.lag1_autocorr - 0.5163, bool(abs(r_.lag1_autocorr - 0.5163) < 0.01)))
        checks.append(("reproduces Exp 12 top-10% share 0.3386", r_.top10_share - 0.3386, bool(abs(r_.top10_share - 0.3386) < 0.01)))

    # scan with common draws
    means = np.round(np.arange(0.40, 0.6601, 0.02), 3) if not FAST else np.array([0.45, 0.55])
    cvs = [0.15, 0.30] if not FAST else [0.30]
    scan = []
    for hname, eps, inst in hosts:
        for cv in cvs:
            s2 = np.log(1 + cv ** 2)
            for mean in means:
                pk = np.exp(np.log(mean) - s2 / 2 + np.sqrt(s2) * Z)
                evaluate("scan", hname, eps, inst, mean, cv, pk, scan)
        stamp(f"14D scan {hname}")
    scan = pd.DataFrame(scan)
    matched = []
    targets = [0.10, 0.20, 0.30, 0.50]
    for hname, eps, inst in hosts:
        for cv in cvs:
            g = scan[(scan.host == hname) & (scan.cv == cv)].sort_values("mean_peak")
            af, mp = g.active_fraction.values, g.mean_peak.values
            s2 = np.log(1 + cv ** 2)
            for tgt in targets:
                j = np.where((af[:-1] <= tgt) & (af[1:] > tgt))[0]
                if len(j) == 0: continue
                j = j[0]; mean = mp[j] + (tgt - af[j]) * (mp[j + 1] - mp[j]) / (af[j + 1] - af[j])
                pk = np.exp(np.log(mean) - s2 / 2 + np.sqrt(s2) * Z)
                row, loss = evaluate("matched", hname, eps, inst, mean, cv, pk, matched)
                row["target_active_fraction"] = tgt
                per_site[f"matched_{hname}_cv{cv}_target{tgt}"] = loss
        stamp(f"14D matched {hname}")
    matched = pd.DataFrame(matched)
    orig.to_csv(os.path.join(d, "D1_original_conditions.csv"), index=False)
    scan.to_csv(os.path.join(d, "D2_scan_common_draws.csv"), index=False)
    matched.to_csv(os.path.join(d, "D3_activity_matched.csv"), index=False)
    np.savez_compressed(os.path.join(d, "per_site_loss_sequences.npz"), **per_site)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for (h, cv), g in scan.groupby(["host", "cv"]):
        g = g.sort_values("active_fraction")
        axes[0].plot(g.active_fraction, g.lag1_autocorr, "o-", ms=3, label=f"{h} CV {cv}")
        axes[1].plot(g.active_fraction, g.burstiness_quiet_runs, "o-", ms=3, label=f"{h} CV {cv}")
        axes[2].plot(g.active_fraction, g.top10_share, "o-", ms=3, label=f"{h} CV {cv}")
    for ax, t in zip(axes, ["lag-1 autocorrelation", "burstiness of quiet runs", "top-10% share"]):
        ax.set_xlabel("active fraction"); ax.set_ylabel(t)
    axes[0].legend(fontsize=6); fig.tight_layout(); fig.savefig(os.path.join(d, "fig14D_01_metrics_vs_activity.png"), dpi=200); plt.close(fig)
    save_checks(d, checks)
    save_json(d, "parameters.json", dict(reference=REF, gamma=GAMMA, delta=0.001, hD=H_D, T=60, sites=NS, intervals=I,
              scan_means=list(means), cvs=cvs, targets=targets, exp12_sequences_used=npz is not None, seed_if_regenerated=20261015))
    with open(os.path.join(d, "README.txt"), "w") as f:
        f.write("14D - Episodic activity with activity-independent clustering measures\n"
                "D1_original_conditions.csv: the eight Experiment 12 conditions with all measures.\n"
                "D2_scan_common_draws.csv: mean peak scanned with common standard-normal draws.\n"
                "D3_activity_matched.csv: each host rerun at the mean giving active fraction 0.1/0.2/0.3/0.5.\n"
                "Measures: active fraction; lag-1 autocorrelation; P(active|active) - P(active|quiet); burstiness of quiet-run\n"
                "lengths (s - m)/(s + m); quiet runs in first/last third; top-10% share; Gini of loss over active intervals.\n"
                "per_site_loss_sequences.npz: per-site loss per interval for every condition.\n")
    save_versions(d); stamp("14D done")


## 14E — Robustness sample in the bistable regime
Latin hypercube of 300 sets around the reference (n, q 3–5; K 0.75–1.05; θ0 0.65–0.95; α 0.6–1.4; r = 1) plus the reference itself.
Per set: folds with and without adaptation; stage boundaries D_irr = u↓/γ, D_spont = u↑/γ; at baseline 0.43 u↑, the treated/naive
critical challenge (ε = 0.03, duration 2/ε) with the model-defined therapy; and the T = 60 ratchet threshold of the ε = 0.01 host
relative to the instant host.


In [ ]:
def run_14E():
    d = outdir("14E_robustness_sample"); stamp("14E start")
    NSET = 300 if not FAST else 12
    X = qmc.scale(qmc.LatinHypercube(d=5, seed=20261018).random(NSET), [3, 3, 0.75, 0.65, 0.6], [5, 5, 1.05, 0.95, 1.4])
    X = np.vstack([[4, 4, 0.9, 0.8, 1.0], X])
    rows = []
    for i, (n, q, K, th0, a) in enumerate(X):
        p = dict(n=n, q=q, K=K, th0=th0, r=1.0, a=a)
        fa, f0 = folds(p), folds(p, a=0.0)
        r = dict(set_id=i, reference=(i == 0), n=n, q=q, K=K, th0=th0, a=a)
        if fa is None or fa[1] <= max(fa[3], 0):
            r.update(bistable=False); rows.append(r); continue
        u_up, u_dn = fa[1], fa[3]
        w0 = (f0[1] - max(f0[3], 0)) if (f0 is not None and f0[1] > 0) else 0.0
        ub = 0.43 * u_up
        r.update(bistable=True, u_up=u_up, u_down=u_dn, width_without_adaptation=w0,
                 bistability_requires_adaptation=bool(w0 < 0.01), stage1_exists=bool(u_dn > 0),
                 D_irr=u_dn / GAMMA if u_dn > 0 else 0.0, D_spont=u_up / GAMMA, baseline=ub,
                 diseased_state_at_baseline=bool(ub > u_dn))
        if ub > u_dn:
            lo, hi = eq_state(p, ub, "low", gamma=0.0), eq_state(p, ub, "high", gamma=0.0)
            r.update(H_low=lo[0], M_low=lo[1], A_low=lo[2], H_high=hi[0], M_high=hi[1], A_high=hi[2])
        rows.append(r)
    S = pd.DataFrame(rows)
    E = S[S.bistable.fillna(False) & S.diseased_state_at_baseline.fillna(False)].copy()
    stamp(f"14E folds done: {len(S)} sets, {len(E)} eligible")
    P = {k: E[k].values for k in ["n", "q", "K", "th0", "a"]}
    pv = dict(n=P["n"], q=P["q"], K=P["K"], th0=P["th0"], r=np.ones(len(E)), a=P["a"])
    N = len(E); ub = E.baseline.values
    low = (E.H_low.values, E.M_low.values, E.A_low.values, np.zeros(N))
    high = (E.H_high.values, E.M_high.values, E.A_high.values, np.zeros(N))
    eps = 0.03; dch = 2 / eps; settle = 10 / eps
    m = Model(**pv, eps=eps); mi = Model(**pv, eps=eps, instant=True)
    treated, ok, ret = therapy(m, high)
    treated = tuple(np.where(ok, treated[k], (low[0], low[1], high[2], low[3])[k]) for k in range(4))
    def crit_for(model, st, sett):
        def conv(U):
            s, _ = model.run(st, schedule(ub, np.zeros(N), np.full(N, dch), U), 0.0, dch + sett)
            return s[0] > 0.5
        return bisect(conv, ub.copy(), np.full(N, 4.0), 14 if not FAST else 6)
    cn, _ = crit_for(m, low, settle); ct, rl = crit_for(m, treated, settle); ci, _ = crit_for(mi, low, 200.0)
    E["therapy_success"] = ok; E["A_retained"] = ret
    E["critical_naive"] = cn; E["critical_treated"] = ct; E["critical_instant"] = ci; E["treated_relapse_without_challenge"] = rl
    E["ratio_treated_naive"] = ct / cn; E["ratio_treated_instant"] = ct / ci
    stamp("14E window done")
    T = 60.0; u_up = E.u_up.values
    def ratchet_thr(model, ncyc):
        def conv(pk):
            umax = pk / (1 - np.exp(-K_REGROW * T)); st = tuple(np.zeros(N) for _ in range(4)); ent = []
            for c in range(ncyc):
                hmin = np.full(N, 9.0)
                def mon(t, H, M, A, D): np.minimum(hmin, H, out=hmin)
                st, _ = model.run(st, lambda t: umax * (1 - np.exp(-K_REGROW * t)), 0.0, T, monitor=mon)
                ent.append(hmin > 0.5)
            return np.all(ent[-3:], axis=0)
        return bisect(conv, 0.5 * u_up, 2.5 * u_up, 12 if not FAST else 5)
    ti, _ = ratchet_thr(mi, 15)
    tm, _ = ratchet_thr(Model(**pv, eps=0.01), 80 if not FAST else 20)
    E["ratchet_instant_T60"] = ti; E["ratchet_eps001_T60"] = tm
    E["ratchet_margin_over_instant"] = tm / ti; E["ratchet_instant_over_static"] = ti / u_up
    stamp("14E ratchet done")
    S = S.merge(E[["set_id", "therapy_success", "A_retained", "critical_naive", "critical_treated", "critical_instant",
                   "treated_relapse_without_challenge", "ratio_treated_naive", "ratio_treated_instant",
                   "ratchet_instant_T60", "ratchet_eps001_T60", "ratchet_margin_over_instant", "ratchet_instant_over_static"]],
                on="set_id", how="left")
    S.to_csv(os.path.join(d, "E_sample_results.csv"), index=False)
    B = S[S.bistable.fillna(False)]
    def q(col, frame=E):
        v = frame[col].dropna(); return f"{v.median():.3f} [{v.quantile(.1):.3f}, {v.quantile(.9):.3f}] (n={len(v)})"
    summ = [("sets drawn (plus reference)", NSET), ("bistable at u >= 0", len(B)),
            ("bistability requires adaptation (width < 0.01 at a = 0)", int(B.bistability_requires_adaptation.sum())),
            ("stage 1 exists (u_down > 0)", int(B.stage1_exists.sum())),
            ("eligible for window (diseased state at baseline)", len(E)),
            ("therapy protocol succeeded", int(E.therapy_success.sum())),
            ("treated/naive ratio, median [10th, 90th]", q("ratio_treated_naive")),
            ("treated/instant ratio", q("ratio_treated_instant")),
            ("fraction with treated/naive < 1", float((E.ratio_treated_naive < 1).mean())),
            ("ratchet margin eps=0.01 / instant", q("ratchet_margin_over_instant")),
            ("instant ratchet threshold / static fold", q("ratchet_instant_over_static")),
            ("D_irr among sets with stage 1", q("D_irr", B[B.stage1_exists == True])),
            ("D_spont", q("D_spont", B))]
    pd.DataFrame(summ, columns=["measure", "value"]).to_csv(os.path.join(d, "E_summary.csv"), index=False)
    ref_row = S[S.reference].T; ref_row.to_csv(os.path.join(d, "E_reference_row.csv"))
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    for axi, col in zip(ax, ["ratio_treated_naive", "ratchet_margin_over_instant", "D_irr"]):
        axi.hist(S[col].dropna(), bins=30); axi.set_xlabel(col)
        if S[S.reference][col].notna().any(): axi.axvline(float(S[S.reference][col].iloc[0]), color="r")
    fig.tight_layout(); fig.savefig(os.path.join(d, "fig14E_01_distributions.png"), dpi=200); plt.close(fig)
    save_json(d, "parameters.json", dict(n=[3, 5], q=[3, 5], K=[0.75, 1.05], th0=[0.65, 0.95], a=[0.6, 1.4], r=1, seed=20261018, sets=NSET,
              baseline="0.43 u_up", eps_window=0.03, duration=dch, settle=settle, ratchet="T = 60, eps = 0.01 (80 cycles) vs instant (15)",
              conversion_ratchet="entrapped in each of the last 3 cycles", gamma=GAMMA))
    with open(os.path.join(d, "README.txt"), "w") as f:
        f.write("14E - Robustness sample in the bistable regime\n"
                "E_sample_results.csv: one row per set (set 0 = reference): folds, stage boundaries, whether bistability needs\n"
                "  adaptation, window ratios (treated_protocol vs naive and vs instant, eps = 0.03, duration 2/eps), ratchet\n"
                "  thresholds at T = 60 and the margin of the eps = 0.01 host over the instant host.\n"
                "E_summary.csv: counts and median [10th, 90th percentile]. Ratchet thresholds use an 80-cycle follow-up,\n"
                "  so thresholds of slow hosts very close to tolerance are slightly overestimated.\n")
    save_versions(d); stamp("14E done")


## 14F — Dose-dependent host: grid check of conversion monotonicity
Second set, A* = H − 1.4H², ε = 0.03, baseline 0.25, challenge 2/ε, settling 10/ε, challenges 0.26–4 in steps of 0.01.


In [ ]:
def run_14F():
    d = outdir("14F_dose_dependent_check"); stamp("14F start")
    a, b, ub, eps = 1.0, 1.4, 0.25, 0.03
    grid = np.round(np.arange(0.26, 4.0001, 0.01 if not FAST else 0.1), 3); N = len(grid)
    m = Model(**SECOND, b=b, eps=eps)
    rows = []
    for hist, branch in [("naive", "low"), ("treated_reset", "high")]:
        lo = eq_state(SECOND, ub, "low", gamma=0.0, a=a, b=b); hi = eq_state(SECOND, ub, "high", gamma=0.0, a=a, b=b)
        st = tile((lo[0], lo[1], lo[2] if hist == "naive" else hi[2], 0.0), N)
        st, _ = m.run(st, const(grid), 0.0, 2 / eps)
        st, _ = m.run(st, const(ub), 0.0, 10 / eps)
        conv = st[0] > 0.5
        first = grid[np.argmax(conv)] if conv.any() else np.nan
        mono = bool(conv.any() and conv[np.argmax(conv):].all())
        rows.append(dict(history=hist, A_start=(lo[2] if hist == "naive" else hi[2]),
                         first_converting_challenge=first, monotone=mono, n_converting=int(conv.sum()), n_grid=N))
    df = pd.DataFrame(rows); df["ratio_to_naive"] = df.first_converting_challenge / df.first_converting_challenge.iloc[0]
    df.to_csv(os.path.join(d, "dose_dependent_monotonicity.csv"), index=False)
    save_checks(d, [("conversion monotone in challenge, both histories", bool(df.monotone.all()), bool(df.monotone.all()))])
    with open(os.path.join(d, "README.txt"), "w") as f:
        f.write("14F - Dose-dependent host (A* = H - 1.4 H^2, second set): conversion on a 0.01 grid of challenges.\n"
                "Exp 11 bisection values: naive 0.358, treated 0.380 (ratio 1.06).\n")
    save_versions(d); stamp("14F done")


In [ ]:
# ---------------------------------------------------------------- run
for key, fn in [("A", run_14A), ("B", run_14B), ("C", run_14C), ("D", run_14D), ("E", run_14E), ("F", run_14F)]:
    if RUN[key]:
        fn()

# all checks in one table
allc = []
for f in glob.glob(os.path.join(OUT_ROOT, "*", "checks.csv")):
    c = pd.read_csv(f); c.insert(0, "module", os.path.basename(os.path.dirname(f))); allc.append(c)
if allc:
    allc = pd.concat(allc); allc.to_csv(os.path.join(OUT_ROOT, "all_checks.csv"), index=False); print(allc.to_string())

zip_name = OUT_ROOT + ".zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_ROOT):
        for fn_ in files:
            z.write(os.path.join(root, fn_))
stamp(f"wrote {zip_name}")
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    pass
